In [3]:
from dotenv import load_dotenv
import os
import requests
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory #sắp bị khai tử thay thế bằng langchain_agents.create_agent
from langchain_classic.schema.runnable import RunnablePassthrough 
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
#from langchain_community.chains import LLMMathChain (class này đã bị khai tử)

In [4]:
load_dotenv()

True

In [5]:
# Khởi tạo tool tính toán 
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [6]:
# Khởi tạo tool thời tiết

@tool
def get_weather(city: str) -> str:
    """
    Get current weather for a given city.
    """

    try:
        api_key = os.getenv("OPENWEATHER_APIKEY")

        url = (
            f"https://api.openweathermap.org/data/2.5/weather"
            f"?q={city}"
            f"&appid={api_key}"
            f"&units=metric"
        )

        response = requests.get(url, timeout=10)
        response.raise_for_status()

        data = response.json()

        weather = data["weather"][0]["description"]
        temp = data["main"]["temp"]

        return (
            f"Current weather in {city}: "
            f"{weather}, {temp}°C, "
            
        )

    except Exception as e:
        return f"Could not retrieve weather data for {city}. Error: {str(e)}"

In [9]:
city = "Hanoi"
api_key = os.getenv("OPENWEATHER_APIKEY")

url = (
    f"https://api.openweathermap.org/data/2.5/weather"
    f"?q={city}"
    f"&appid={api_key}"
    f"&units=metric"
)

print(url)

response = requests.get(url)

print(response.status_code)
print(response.text)

https://api.openweathermap.org/data/2.5/weather?q=Hanoi&appid=5a1789e6bbbe5be35fc00f4b2e3585c0&units=metric
200
{"coord":{"lon":105.8412,"lat":21.0245},"weather":[{"id":502,"main":"Rain","description":"heavy intensity rain","icon":"10n"}],"base":"stations","main":{"temp":30.92,"feels_like":37.09,"temp_min":30.92,"temp_max":30.92,"pressure":1003,"humidity":69,"sea_level":1003,"grnd_level":1003},"visibility":10000,"wind":{"speed":4,"deg":137,"gust":7.25},"rain":{"1h":5.62},"clouds":{"all":100},"dt":1781530016,"sys":{"country":"VN","sunrise":1781475303,"sunset":1781523547},"timezone":25200,"id":1581130,"name":"Hanoi","cod":200}


In [10]:
# Khởi tạo mô hình LLM
llm = ChatGoogleGenerativeAI(model = 'gemma-4-31b-it', temperature = 1)

# Khởi tạo bộ nhớ
memory = ConversationBufferMemory(
    return_messages = True,
    memory_key = "chat_history"
)
# Thêm tools vào agent 
tools = [calc, get_weather]

# Cấu hình prompt 
conversation_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant."),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{input}"),

    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Khởi tạo agent
agent = create_tool_calling_agent(llm, tools, conversation_prompt)

agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True 
)

C:\Users\toang\AppData\Local\Temp\ipykernel_7000\402863418.py:5: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


In [11]:
def chat(message):
    response = agent_executor.invoke({
        "input": message,
        "chat_history": memory.load_memory_variables({})["chat_history"]
    })

    memory.save_context(
        {"input": message},
        {"output": response["output"]}
    )
    return response["output"]

In [12]:
chat("what is the weather like in Hà Nội")



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Hà Nội'}`
responded: [{'type': 'thinking', 'thinking': 'The user is asking for the weather in Hà Nội. I have a tool called `get_weather` that can provide current weather for a given city. I should call this tool with the city "Hà Nội".', 'index': 0}]

Current weather in Hà Nội: heavy intensity rain, 30.26°C, The current weather in Hà Nội is 30.26°C with heavy rain.

> Finished chain.


'The current weather in Hà Nội is 30.26°C with heavy rain.'